# Week 1 — MNIST Classification with PyTorch
### Diffusion Models from Scratch — SoC 2026

Welcome! This notebook walks you through building your first PyTorch classifier from scratch.

**How this notebook works:**
- Sections marked **`# TODO`** are blanks for you to fill in.
- Each TODO has a hint pointing to a specific section of the Week 1 resources. Read them — don't guess.
- Conceptual question cells (marked **❓**) are short-answer; type your answer in the markdown cell below.
- At the end, an assertion will check your test accuracy. Don't modify the assertion.

**Setup (do this first):**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Run cells top to bottom with `Shift+Enter`.

**Submission:** Push this completed notebook to your GitHub repo along with a `README.md` (template at the end). Due before Saturday's group call.

**Honor code:** Don't paste in another mentee's answers, and don't use LLMs to fill in the blanks. The point is to internalize this material — Week 2+ depends on it. If you're stuck, ask in the group chat or DM your mentor.


## Section 0 — Imports and reproducibility
*(Provided for you. Run this cell.)*

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Section 1 — Device setup

**TODO 1:** Set `device` to `"cuda"` if a GPU is available, otherwise `"cpu"`.

*Hint: see the "Tensors" section of the [60-Minute Blitz](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html) — search for `torch.cuda.is_available`.*

In [3]:
# TODO 1: define `device`
device = "cuda"  # replace ___
print (f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {device}")
assert str(device) in {"cuda", "cpu"}, "device must be 'cuda' or 'cpu'"


CUDA available: True
Using device: cuda


## Section 2 — Custom Dataset class

The deliverable says: *"don't just use `torchvision.datasets.MNIST` directly — subclass it or wrap it."* We'll wrap it. Wrapping gives you a place to add custom preprocessing or augmentation later (which you'll need for diffusion models in a few weeks).

**TODO 2a:** Complete `__len__` — it should return the number of samples in `self.mnist`.

**TODO 2b:** Complete `__getitem__` — it should return `(image_tensor, label)` for the sample at index `idx`.

*Hints:*
- *Read the [Dataset docs](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) — there's a tiny example showing exactly what these two methods do.*
- *`self.mnist[idx]` already returns `(image, label)` because we passed a transform — you can just forward it.*

In [5]:
class MNISTWrapper(Dataset):
    def __init__(self, root="./data", train=True, download=True):
        # Standard MNIST normalization (mean=0.1307, std=0.3081 are well-known constants)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        self.mnist = datasets.MNIST(
            root=root, train=train, download=download, transform=self.transform,
        )

    def __len__(self):
        return (self.mnist).__len__()

    def __getitem__(self, idx):
        return self.mnist[idx]

### ❓ Conceptual Question 1
**How does a `DataLoader` differ from just having a Python list of batches?** Give at least three concrete differences.

*(Write your answer in the cell below by replacing the placeholder text.)*

**Your answer:**

*(replace this text)*

## Section 3 — Data splits and DataLoaders
*(Provided. Run this cell once you've completed Section 2 — it sanity-checks your Dataset class.)*

In [6]:
BATCH_SIZE = 128
VAL_FRACTION = 0.1

full_train = MNISTWrapper(train=True, download=True)
test_set   = MNISTWrapper(train=False, download=True)

# Sanity check: your Dataset class should return one (image, label) per index
img, lbl = full_train[0]
assert img.shape == (1, 28, 28), f"Expected (1,28,28), got {img.shape}"
assert isinstance(lbl, int) and 0 <= lbl <= 9, f"Label looks wrong: {lbl}"
print(f"Sample 0: image shape {img.shape}, label {lbl}  ✓")

val_size   = int(len(full_train) * VAL_FRACTION)
train_size = len(full_train) - val_size
train_set, val_set = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

100.0%
100.0%
100.0%
100.0%

Sample 0: image shape torch.Size([1, 28, 28]), label 5  ✓
Train: 54000 | Val: 6000 | Test: 10000


## Section 4 — Model definition

**TODO 3:** Complete the MLP architecture below. The structure should be:
1. Flatten the input (shape `(B,1,28,28)` → `(B,784)`)
2. Linear `in_dim → hidden`, then ReLU, then Dropout
3. Linear `hidden → hidden`, then ReLU, then Dropout
4. Linear `hidden → out_dim` (raw logits, no activation)

*Hints:*
- *See [`nn.Sequential` docs](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) and the "Build the Neural Network" section of the 60-Minute Blitz.*
- *Each layer type: [`nn.Flatten`](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html), [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), [`nn.ReLU`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html), [`nn.Dropout`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html).*

**TODO 4:** ⚠️ **Read carefully.** Why does the final layer output raw logits (no softmax)? Hint: look at the docs for [`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) — pay attention to the "Note" about LogSoftmax. *Write your answer in the markdown cell after the code.*

In [9]:
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, p_drop=0.2):
        super().__init__()
        # TODO 3: build the architecture described above using nn.Sequential
        self.net = nn.Sequential(
            nn.Flatten(start_dim=1, end_dim=-1),  # we want to flatten all except batch dimension
            nn.Linear(in_features=in_dim, out_features=hidden),
            nn.ReLU(),
            nn.Dropout(p=p_drop),

            nn.Linear(in_features=hidden, out_features=hidden),
            nn.ReLU(),
            nn.Dropout(p=p_drop),

            nn.Linear(in_features=hidden, out_features=out_dim)

        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Sanity check: forward a dummy batch
dummy = torch.randn(4, 1, 28, 28, device=device)
out = model(dummy)
assert out.shape == (4, 10), f"Expected logits of shape (4,10), got {out.shape}"
print(f"Forward pass OK: input {dummy.shape} → output {out.shape}  ✓")


MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=256, out_features=10, bias=True)
  )
)
Total parameters: 269,322
Forward pass OK: input torch.Size([4, 1, 28, 28]) → output torch.Size([4, 10])  ✓


**TODO 4 — Your answer (why no softmax at the end?):**

From the documentation at the given link, `nn.CrossEntropyLoss` expects the model output to be **raw logits** (unnormalized scores), not probabilities. The logits do not need to be positive and do not need to sum to 1.


`CrossEntropyLoss` internally applies `LogSoftmax` before computing the negative log-likelihood loss. Therefore, we should **not** add a `Softmax` layer at the end of the network, since that would be redundant and can reduce numerical stability.

The final layer should therefore output raw logits:

```python
nn.Linear(hidden, out_dim)

## Section 5 — Loss and optimizer
*(Provided.)*

In [10]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Section 6 — Evaluation helper

You'll call this both during training (for validation) and at the end (for the test set).

**TODO 5:** Write the full `evaluate` function.

Requirements:
- Decorate with `@torch.no_grad()` so no graph is built
- Call `model.eval()` so dropout/batchnorm behave correctly
- Loop over `loader`, move `x, y` to `device`, compute logits and loss
- Track **weighted** loss (multiply by batch size, divide by total samples at the end) so the average is correct even if the last batch is smaller
- Track number of correct predictions for accuracy
- Return `(average_loss, accuracy)` as floats

*Hints:*
- *Karpathy video: he discusses why `model.eval()` matters for layers like dropout around the 1h05m mark.*
- *PyTorch docs on [`torch.no_grad`](https://pytorch.org/docs/stable/generated/torch.no_grad.html).*
- *Predictions = `logits.argmax(dim=1)` (the class with the highest logit).*

In [ ]:
# TODO 5: write the evaluate function
@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    loss_to_be_accumulated = 0.0
    correctones = 0
    totalsamples =0

    for x, y in loader:
        x = x.to(device)
        y= y.to (device)

        logits = model(x)
        loss = loss_fn(logits, y) #this is going to be the average over batch size, so we need to multiply it to add to the accumul. loss
        batch_size = x.size(0)


### ❓ Conceptual Question 2
**When and why do we use `torch.no_grad()`?** What goes wrong if you forget it during evaluation?

(One paragraph is fine.)

**Your answer:**

*(replace this text)*

## Section 7 — Training loop

This is the heart of the week. Internalize the 5-step training rhythm; you'll write it hundreds of times.

**TODO 6:** Fill in the **5 steps** of the inner training step (the lines marked `# step 1` through `# step 5`).

The steps, in order:
1. Clear stale gradients on the optimizer
2. Forward pass — compute logits from `x`
3. Compute the loss from `logits` and `y`
4. Backward pass — populate `.grad` on every parameter
5. Optimizer step — update parameters using their gradients

*Hints:*
- *Karpathy's micrograd video (~1h00m) shows exactly this loop being built from scratch.*
- *60-Minute Blitz, "Training a Classifier" section, shows the same 5 lines.*

In [ ]:
NUM_EPOCHS = 10
CKPT_PATH  = "best_model.pt"
best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_samples = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        # TODO 6: the 5-step training rhythm
        ___   # step 1: clear stale gradients
        ___   # step 2: forward pass → logits
        ___   # step 3: compute loss
        ___   # step 4: backward pass
        ___   # step 5: optimizer step

        # (provided) running metrics
        running_loss    += loss.item() * x.size(0)
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_samples += x.size(0)

    train_loss = running_loss / running_samples
    train_acc  = running_correct / running_samples
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    # TODO 7: save a checkpoint ONLY when val_acc improves
    # Save a dict containing: epoch, model_state, optimizer_state, val_acc
    # Use torch.save(...) to write it to CKPT_PATH.
    # Hint: PyTorch saving/loading tutorial → https://pytorch.org/tutorials/beginner/saving_loading_models.html
    if ___:
        best_val_acc = ___
        torch.save({
            ___
        }, CKPT_PATH)
        print(f"  -> new best val_acc; checkpoint saved to {CKPT_PATH}")


### ❓ Conceptual Question 3
**Try removing `optimizer.zero_grad()`** from your training loop (just for this question — put it back after!). Re-run training for 2 epochs and observe what happens to the loss.

Then answer: **why does forgetting `zero_grad()` break training?** What does `loss.backward()` actually *do* to the `.grad` attribute — does it overwrite or accumulate?

**Your answer:**

*(replace this text — include what you observed when you removed `zero_grad()`)*

### ❓ Conceptual Question 4
**What's the difference between `model.train()` and `model.eval()`?** Name at least one layer type whose behavior changes, and explain *how* it changes.

**Your answer:**

*(replace this text)*

## Section 8 — Reload checkpoint and evaluate on test set

**TODO 8:** Load the checkpoint from `CKPT_PATH` and use it to evaluate test accuracy.

Steps:
1. Load the checkpoint dict with `torch.load(...)` (pass `map_location=device`)
2. Create a *fresh* `MLP()` instance and move it to `device`
3. Load the saved weights into it using `load_state_dict(...)`
4. Call your `evaluate(...)` function on `test_loader`

*Why a fresh model?* This tests that your save/load actually round-trips correctly — if you just reused `model`, you'd be testing the in-memory weights, not the checkpoint.

*Hint: [Saving & Loading Models tutorial](https://pytorch.org/tutorials/beginner/saving_loading_models.html).*

In [ ]:
# TODO 8: reload checkpoint and evaluate on the test set
ckpt = ___
fresh_model = ___
fresh_model.___

test_loss, test_acc = ___
print(f"\nLoaded checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")
print(f"FINAL TEST ACCURACY: {test_acc:.4f}")

# DO NOT MODIFY: this is the deliverable threshold
assert test_acc >= 0.97, f"Test accuracy {test_acc:.4f} below required 0.97"
print("Deliverable threshold met. 🎉")


## Section 9 — Plot training curves
*(Provided. Sanity-check that loss decreases and val tracks train.)*

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"],   label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["val_acc"],   label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Section 10 — Final reflection

### ❓ Conceptual Question 5
Look at your training/validation curves above. Answer briefly:
1. Do you see signs of **overfitting** (train accuracy much higher than val)? If yes, when does it start?
2. What's one change you could make to reduce overfitting?
3. If you had a GPU budget of 100 epochs instead of 10, would you just train longer? Why or why not?

**Your answer:**

*(replace this text)*

## README template

Copy the following into a `README.md` file in your repo root. Fill in the bracketed sections.

```markdown
# Week 1 — MNIST Classification

**Mentee:** [Your name]
**SoC Track:** Diffusion Models from Scratch — SoC 2026

## Final results
- **Test accuracy:** [X.XX%]
- **Best validation accuracy:** [X.XX%] at epoch [N]
- **Final train loss:** [X.XXXX]

## Design choices
- **Architecture:** [brief description — layers, hidden size, dropout rate]
- **Optimizer:** [Adam with lr=1e-3 / whatever you used]
- **Batch size:** [X]
- **Epochs trained:** [X]
- **Validation split:** [X% of training data, seed=42]

## What I learned
[2-4 sentences on the biggest takeaway from this week]

## What I'd do differently
[1-2 sentences — e.g. try a CNN, longer training, different LR]

## How to reproduce
1. Open `week1_mnist.ipynb` in Colab with a T4 GPU runtime.
2. Run all cells top to bottom.
3. Checkpoint will be saved to `best_model.pt`.
```

---

**You're done!** Commit and push:
```
git add week1_mnist.ipynb README.md best_model.pt
git commit -m "Week 1: MNIST classifier (test acc: XX.XX%)"
git push
```

(Note: `best_model.pt` is small for MLPs but for larger models in later weeks, you'll want to add weights to `.gitignore` and document where they live instead.)
